In [ ]:
import os
import json
import re
import time
import logging
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq  # Fixed for Groq SDK

# =========================
# Logging Configuration
# =========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("extraction_process_groq.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# Groq client setup
# =========================
load_dotenv()
API_KEY = os.getenv("GROQ_API_KEY")
if not API_KEY:
    logger.error("Set GROQ_API_KEY in your environment or .env")
    raise RuntimeError("Missing API Key")

# Initialize the Groq client
client = Groq(api_key=API_KEY)

# Using the high-performance Llama 3.3 70B model on Groq
MODEL_ID = "llama-3.3-70b-versatile"

# =========================
# Prompts (UNCHANGED)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 to 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
5. Each keyword should ideally be 1-3 words, but can be longer if it is a specific phrase but in rare cases.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# =========================
# Helpers
# =========================
def _extract_json(text: str) -> Dict[str, Any]:
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            block = m.group(0)
            return json.loads(block)
    except Exception as e:
        logger.warning(f"Failed to parse JSON from output: {e}")
    return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if k and k.lower() not in seen:
            seen.add(k.lower())
            out.append(k)
    return out

def _authors_block(item: Dict[str, Any]) -> str:
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af: parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

def _call_groq(messages: List[Dict[str, str]], max_new_tokens: int = 512, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=0.0,
                max_tokens=max_new_tokens,
            )
            # The structure for Groq is identical to OpenAI/OpenRouter choices
            if hasattr(resp, 'choices') and len(resp.choices) > 0:
                return resp.choices[0].message.content or ""
        except Exception as e:
            logger.error(f"Attempt {attempt+1} failed: {e}")
            if "401" in str(e):
                logger.error("Authentication Error. Check GROQ_API_KEY.")
            
            # Groq often gives 429 for rate limits; wait accordingly
            if "429" in str(e):
                logger.warning("Rate limit hit. Waiting 10 seconds...")
                time.sleep(10)
            elif attempt < retries - 1:
                time.sleep(5) 
    return ""

def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    user_content = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    messages = [
        {"role": "system", "content": System_Prompt.strip()},
        {"role": "user", "content": user_content.strip()},
    ]
    raw_output = _call_groq(messages)
    data = _extract_json(raw_output)
    return {
        "keywords": _clean_list(data.get("keywords", [])),
        "countries": _clean_list(data.get("countries", []))
    }

# =========================
# Folder Processing Logic
# =========================
def run_batch_extraction(input_folder: str, output_folder: str):
    if not os.path.exists(input_folder):
        logger.error(f"Input folder '{input_folder}' does not exist.")
        return

    os.makedirs(output_folder, exist_ok=True)
    json_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.json')])
    
    if not json_files:
        logger.info(f"No JSON files found in {input_folder}")
        return

    logger.info(f"Starting Groq batch process for {len(json_files)} files.")
    master_results = []

    for file_name in json_files:
        input_path = os.path.join(input_folder, file_name)
        logger.info(f"--- Processing File: {file_name} ---")
        
        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            file_results = []
            for i, item in enumerate(data):
                title = item.get("title") or item.get("topic") or "No Title"
                abstract = item.get("abstract") or ""
                authors = _authors_block(item)
                
                logger.info(f"[{i+1}/{len(data)}] Extracting: {title[:40]}...")
                extracted = extract_keywords_and_countries(title, abstract, authors)
                
                file_results.append({
                    "title": title,
                    "authors": authors,
                    "keywords": extracted["keywords"],
                    "countries": extracted["countries"],
                    "source_file": file_name,
                    "Year": 2023 
                })
                # Groq is extremely fast, so adding a tiny delay to avoid hitting Rate Limits (RPM)
                time.sleep(0.5) 
            
            base_name = os.path.splitext(file_name)[0]
            individual_out = os.path.join(output_folder, f"{base_name}_extracted_groq.json")
            with open(individual_out, "w", encoding="utf-8") as f:
                json.dump(file_results, f, indent=2)
            
            master_results.extend(file_results)
            logger.info(f"Finished {file_name}. Saved to {individual_out}")

        except Exception as e:
            logger.error(f"Failed to process {file_name}: {e}")

    master_out = os.path.join(output_folder, "final_combined_keywords_groq.json")
    with open(master_out, "w", encoding="utf-8") as f:
        json.dump(master_results, f, indent=2)
    
    logger.info(f"BATCH COMPLETE. Final master file: {master_out}")

if __name__ == "__main__":
    # Ensure these folders exist
    INPUT_DIR = "dummy" 
    OUTPUT_DIR = "groq/extracted_results"

    run_batch_extraction(INPUT_DIR, OUTPUT_DIR)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'raw_abstracts'

THis is code version 2

In [1]:
import os
import json
import re
import time
import logging
from typing import List, Dict, Any
from datetime import datetime, timedelta
from dotenv import load_dotenv
from groq import Groq

# =========================
# Logging Configuration
# =========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("extraction_process_groq.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# Groq Client & Rate Limits
# =========================
load_dotenv()
API_KEY = os.getenv("GROQ_API_KEY")
if not API_KEY:
    logger.error("Set GROQ_API_KEY in your environment or .env")
    raise RuntimeError("Missing API Key")

client = Groq(api_key=API_KEY)
MODEL_ID = "openai/gpt-oss-120b"

class GroqLimiter:
    """Strictly manages Requests Per Minute (RPM) to avoid 429 errors."""
    def __init__(self, max_rpm=10): 
        self.max_rpm = max_rpm
        self.request_history = []

    def throttle(self):
        now = datetime.now()
        self.request_history = [t for t in self.request_history if now - t < timedelta(seconds=60)]
        if len(self.request_history) >= self.max_rpm:
            wait_time = 60 - (now - self.request_history[0]).total_seconds()
            if wait_time > 0:
                logger.info(f"Rate limit buffer: Sleeping for {wait_time:.2f}s...")
                time.sleep(wait_time + 0.2)
        self.request_history.append(datetime.now())

limiter = GroqLimiter()

# =========================
# Prompts (EXACT - UNCHANGED)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
5. Each keyword should ideally be 1-3 words, but can be longer if it is a specific phrase but in rare cases.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# =========================
# Helpers
# =========================
def _extract_json(text: str) -> Dict[str, Any]:
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m: return json.loads(m.group(0))
    except Exception as e:
        logger.warning(f"JSON Parse Failed: {e}")
    return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if k and k.lower() not in seen:
            seen.add(k.lower())
            out.append(k)
    return out

def _authors_block(item: Dict[str, Any]) -> str:
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm, af = (a.get("name") or "").strip(), (a.get("affiliation") or "").strip()
            if nm or af: parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

def _call_groq(messages: List[Dict[str, str]], retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            limiter.throttle()
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                temperature=0.0,
                max_tokens=512,
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            logger.error(f"Attempt {attempt+1} Error: {e}")
            if "429" in str(e):
                logger.warning("Rate limit hit! Sleeping 20s...")
                time.sleep(20)
            elif attempt < retries - 1:
                time.sleep(5) 
    return ""

# =========================
# Main Execution Logic
# =========================
def run_batch_extraction(input_folder: str, output_folder: str):
    if not os.path.exists(input_folder):
        logger.error(f"Input folder '{input_folder}' not found.")
        return

    os.makedirs(output_folder, exist_ok=True)
    json_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.json')])
    master_results = []

    for file_name in json_files:
        input_path = os.path.join(input_folder, file_name)
        logger.info(f"--- File: {file_name} ---")
        
        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            file_results = []
            for i, item in enumerate(data):
                # FIXED: Case-insensitive Title Search
                title = next((item[k] for k in item if k.lower() in ["title", "topic", "paper_title"]), "No Title")
                
                abstract = item.get("abstract") or ""
                authors = _authors_block(item)
                
                logger.info(f"[{i+1}/{len(data)}] Extracting: {str(title)[:40]}...")
                
                user_content = User_Prompt_template.format(
                    few_shot=FEW_SHOT.strip(),
                    title_or_topic=str(title).strip(),
                    authors_block=authors.strip(),
                    abstract_text=abstract.strip()
                )
                
                messages = [
                    {"role": "system", "content": System_Prompt.strip()},
                    {"role": "user", "content": user_content.strip()},
                ]
                
                raw_output = _call_groq(messages)
                parsed = _extract_json(raw_output)
                
                res_entry = {
                    "title": title,
                    "authors": authors,
                    "keywords": _clean_list(parsed.get("keywords", [])),
                    "countries": _clean_list(parsed.get("countries", [])),
                    "source_file": file_name,
                    "Year": 2023 
                }
                file_results.append(res_entry)
                master_results.append(res_entry)

            # Individual File Save
            out_name = f"{os.path.splitext(file_name)[0]}_extracted_groq.json"
            with open(os.path.join(output_folder, out_name), "w", encoding="utf-8") as f:
                json.dump(file_results, f, indent=2)

        except Exception as e:
            logger.error(f"Critical error in {file_name}: {e}")

    # Master Save
    with open(os.path.join(output_folder, "final_combined_keywords_groq.json"), "w", encoding="utf-8") as f:
        json.dump(master_results, f, indent=2)
    logger.info("BATCH PROCESS COMPLETE.")

if __name__ == "__main__":
    run_batch_extraction("aiche_file", "groq/extracted_results")

2026-02-15 18:39:03,505 - INFO - --- File: aiche_papers_3298.json ---
2026-02-15 18:39:03,522 - INFO - [1/584] Extracting: 179a- Modeling of Interfacial Properties...
2026-02-15 18:39:04,373 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 18:39:04,387 - INFO - [2/584] Extracting: 179b- Modeling the Viscosity of Imidazol...
2026-02-15 18:39:05,088 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 18:39:05,095 - INFO - [3/584] Extracting: 179c- Techniques for Measuring and Model...
2026-02-15 18:39:06,053 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 18:39:06,102 - INFO - [4/584] Extracting: 179d- Thermophysical Properties of Binar...
2026-02-15 18:39:06,824 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-15 18:39:06,824 - INFO - [5/584] Extracting: 179f- High Pressu

KeyboardInterrupt: 

This is the version containing multiple accounts
## Work on this currently

In [7]:
import os
import json
import re
import time
import logging
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq

# =========================
# Logging Configuration
# =========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("quota_monitor.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# =========================
# Key & Model Configuration
# =========================
load_dotenv()
# Fetches keys: GROQ_API_KEY, GROQ_API_KEY1, GROQ_API_KEY2, GROQ_API_KEY3, GROQ_API_KEY4
API_KEYS = [os.getenv(f"GROQ_API_KEY{i}" if i > 0 else "GROQ_API_KEY") for i in range(5)]
API_KEYS = [k for k in API_KEYS if k] 

# Failover Models
MODELS = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b"]

if not API_KEYS:
    raise RuntimeError("No API keys found. Ensure GROQ_API_KEY through GROQ_API_KEY4 are in .env")

# =========================
# Prompts (EXACT - UNCHANGED)
# =========================
System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 to 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
3. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
4. Be formatted as a JSON list of exactly 5 to 10 unique entries without explanation.
5. Each keyword should ideally be 1-3 words, but can be longer if it is a specific phrase but in rare cases.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 to 10 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

# =========================
# Failover & Quota Engine
# =========================
class FailoverMonitor:
    def __init__(self, keys: List[str]):
        self.keys = keys
        self.current_key_index = 0

    def get_client(self):
        return Groq(api_key=self.keys[self.current_key_index])

    def call_with_quota_check(self, messages):
        # Cycle through keys
        for _ in range(len(self.keys)):
            client = self.get_client()
            
            # Cycle through models per key
            for model in MODELS:
                try:
                    # Request with raw response to capture headers
                    raw_res = client.chat.completions.with_raw_response.create(
                        model=model,
                        messages=messages,
                        temperature=0.0,
                        max_tokens=512
                    )
                    
                    completion = raw_res.parse()
                    headers = raw_res.headers
                    
                    # Log Live Quota Data
                    rem_req = headers.get("x-ratelimit-remaining-requests", "N/A")
                    rem_tok = headers.get("x-ratelimit-remaining-tokens", "N/A")
                    reset_time = headers.get("x-ratelimit-reset-requests", "N/A")
                    
                    logger.info(f"Success! [Key {self.current_key_index} | Model: {model}]")
                    logger.info(f"Quota Status: {rem_req} req left | {rem_tok} tokens left | Reset: {reset_time}")
                    
                    return completion.choices[0].message.content
                    
                except Exception as e:
                    if "429" in str(e):
                        logger.warning(f"Rate Limit hit for Key {self.current_key_index} ({model}). Switching model...")
                        continue 
                    else:
                        logger.error(f"Unexpected API error: {e}")
                        time.sleep(2)

            # Both models failed on this key
            logger.error(f"KEY {self.current_key_index} FULLY THROTTLED. Moving to next account...")
            self.current_key_index = (self.current_key_index + 1) % len(self.keys)
            time.sleep(1)

        logger.critical("ALL ACCOUNTS AND MODELS REACHED DAILY/MINUTE LIMITS. Sleeping 60s...")
        time.sleep(60)
        return self.call_with_quota_check(messages)

engine = FailoverMonitor(API_KEYS)

# =========================
# Helpers
# =========================
def _extract_json(text):
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        return json.loads(m.group(0)) if m else {}
    except: return {}

def _authors_block(item):
    parts = []
    for a in item.get("authors_structured", []) or []:
        nm, af = (a.get("name") or "").strip(), (a.get("affiliation") or "").strip()
        if nm or af: parts.append(f"{nm} ({af})")
    return "; ".join(parts)[:800]

# =========================
# Batch Logic (UNCHANGED FILE SAVING)
# =========================
def run_batch_extraction(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    json_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.json')])

    for file_name in json_files:
        out_path = os.path.join(output_folder, f"ext_{file_name}")
        if os.path.exists(out_path): continue 

        logger.info(f"--- Processing: {file_name} ---")
        with open(os.path.join(input_folder, file_name), "r", encoding="utf-8") as f:
            data = json.load(f)

        file_results = []
        for i, item in enumerate(data):
            title = next((item[k] for k in item if k.lower() in ["title", "topic"]), "No Title")
            logger.info(f"[{i+1}/{len(data)}] Extracting: {str(title)[:35]}...")
            
            raw_text = engine.call_with_quota_check([
                {"role": "system", "content": System_Prompt.strip()},
                {"role": "user", "content": User_Prompt_template.format(
                    few_shot=FEW_SHOT.strip(), title_or_topic=title,
                    authors_block=_authors_block(item), abstract_text=item.get("abstract", "")
                ).strip()}
            ])
            
            parsed = _extract_json(raw_text)
            file_results.append({
                "title": title, "keywords": parsed.get("keywords", []),
                "countries": parsed.get("countries", []), "source_file": file_name,
                "Year": 2023
            })

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(file_results, f, indent=2)
        logger.info(f"Checkpoint saved: {out_path}")

if __name__ == "__main__":
    run_batch_extraction("aiche_file_output", "groq/extracted_results")